In [ ]:
##background:
#I assume that the prior year loss run is carried forward from last year, rather than being provided again this year. If it were provided again this year, I would compare to an existing file from last year.
#I run this exercise after the triangle exercise #2. This is relevant only because I take the output file from that exercise as the input file for this exercise.
#I wanted to try out fuzzyjoin since I think it has better performance on weirder joins compared to dplyr

##user inputs:
#source data: go to the notebook cell headed with "##create worbook for output, test saving a copy"
#selects: go to the notebook cell headed with "##make selects so that they can easily be read in the notebook"; specify input file, input sheets, and output file
#packages: if you don't have them, install the packages under "## packages, run once globally"


##TODO: Store long-term history.
##TODO: Try to have pattern recognition on distribution of losses to look for anomalies. Probably not appropriate for this small dataset but would make sense for a larger one.
##TODO: plots/markdown?

##Approach: I think the more standard approach is to use dataframes for everything and then look at them in pivoted representation. I want to try keeping things in matrix form.

In [1]:
## packages, run once globally
# install.packages("openxlsx2")
# install.packages("lubridate")
# install.packages("tidyverse")
# install.packages("zoo")
# install.packages("fuzzyjoin")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘stringdist’, ‘geosphere’




In [8]:
##create worbook for output, test saving a copy
library(openxlsx2)
input_path = "output/PYRQ_output.xlsx"
output_path = "output/PYRQ_output.xlsx"
input_sheet_current = "ex1_curr"
input_sheet_prior = "ex1_prior"

wb <- wb_load(input_path)
wb_save(wb,output_path,overwrite = TRUE)

#create styles
dec3="0.000"
dec0="#,##0"
decY="###0"

#function to create a sheet destructively; can use this to clear a sheet also
newsheet_destructive <- function(name) {
    if (name %in% wb$sheet_names) wb$remove_worksheet(sheet=name)
    wb$add_worksheet(name, tabColour = "pink")
}

#function to populate a sheet
popsheet <- function(name,frame,title,style,titlerow,startcolumn) {
    wb$add_data(sheet = name, title, start_row = titlerow, start_col = startcolumn, row_names=FALSE, na="")
    wb$add_data(sheet = name, frame, start_row = titlerow+2, start_col = startcolumn, row_names=TRUE, na="")
    wb$add_numfmt(sheet=name, numfmt=style, 
         dims=wb_dims(rows=(titlerow+3):(nrow(frame)+(titlerow+3)), cols=(startcolumn+1):(ncol(frame)+startcolumn+1)))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow+3):(nrow(frame)+(titlerow+3)),cols=startcolumn:startcolumn))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow):(titlerow+2),cols=startcolumn:(ncol(frame)+startcolumn+1)))
}

#function to get max row and column
bounds <- function(name) {
  ws <- wb$worksheets[[which(wb$sheet_names == name)]]
  cc <- ws$sheet_data$cc
  max_row <- max(as.integer(cc$row_r))
  max_col <- max(col2int(cc$c_r))
  return(c(max_row, max_col))
}

#function to set column widths
autowidth <- function(name,col=-1,w=-1) {
    if (col==-1) ncols=bounds(name)[2] else ncols=col
    if (w == -1) width="auto" else width=w
    wb$set_col_widths(name, cols=1:ncols, widths=width)
}

In [6]:
##look at list of worksheets
sheets <- wb_get_sheet_names(wb)
data.frame(index = seq_along(sheets), name = sheets)


,index,name
,<int>,<chr>
ex1_curr,1,ex1_curr
ex1_prior,2,ex1_prior
ex2_tri,3,ex2_tri
initial diagnostics,4,initial diagnostics
triangles,5,triangles
notes,6,notes


In [11]:
##read the triangle, create matrix, create variables to be used for checks

library(lubridate)
library(tidyverse)
library(fuzzyjoin)

#I don't want scientific notation in output
options(scipen=999)

#load the loss runs
c <- wb_to_df(wb, sheet=input_sheet_current)
c = cbind(c,matrix(input_sheet_current,nrow(c),1))
p <- wb_to_df(wb, sheet=input_sheet_prior)
p = cbind(p,matrix(input_sheet_current,nrow(p),1))
c
p

#s = rbind(c,p)
#s

,claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,"matrix(input_sheet_current, nrow(c), 1)"
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
2,A0001,2019-02-03,2019-06-01,2021-01-21,Closed,72568,72568,0,ex1_curr
3,A0002,2019-06-13,2020-07-31,NA,Open,35287,27458,7829,ex1_curr
4,A0003,2019-08-27,2020-10-05,NA,Open,86564,78926,8638,ex1_curr
5,A0004,2019-09-13,2021-03-27,2022-02-05,Closed,7568,7568,0,ex1_curr
6,A0005,2020-04-11,2021-10-20,2022-06-08,Closed,256845,256845,0,ex1_curr
7,A0006,2020-06-02,2020-06-12,2020-09-29,Closed,43718,43718,0,ex1_curr
8,A0007,2020-08-21,2022-07-03,2022-07-15,Closed,7843,7843,0,ex1_curr
9,A0008,2021-01-07,2021-02-18,NA,Open,78300,23792,54508,ex1_curr
10,A0009,2021-01-20,2021-05-09,NA,Open,87824,79087,8737,ex1_curr


,claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,"matrix(input_sheet_current, nrow(p), 1)"
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
2,A0001,2019-02-03,2019-06-01,2021-01-21,Closed,74570,74570,0,ex1_curr
3,A0002,2019-06-13,2020-07-31,2020-10-15,Closed,27458,27458,0,ex1_curr
4,A0003,2019-08-27,2020-10-05,NA,Open,86564,77926,8638,ex1_curr
5,A0004,2019-09-13,2021-03-27,NA,Open,9811,5464,4347,ex1_curr
6,A0005,2020-04-11,2021-10-20,NA,Open,85426,18129,67297,ex1_curr
7,A0006,2020-06-02,2020-06-12,2020-09-29,Closed,43718,43718,0,ex1_curr
8,A0007,2020-08-11,2022-07-03,2022-07-15,Closed,7843,7843,0,ex1_curr
9,A0008,2021-01-07,2021-02-18,NA,Open,78300,25792,52508,ex1_curr
10,A0009,2021-01-20,2021-05-09,NA,Open,87824,79087,8737,ex1_curr
